# 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# SMS Spam 데이터 다운로드 및 로드
url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv"
df = pd.read_table(url, header=None, names=['label', 'message'])
df = df.reset_index(drop=True)        # 인덱스 초기화

# 데이터 확인
print("데이터셋 크기:", df.shape)
df.head()

In [ ]:
# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
!nvidia-smi

# 데이터 분할

In [ ]:
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt') # 문장 토큰화(sentence tokenization)용 데이터 패키지
nltk.download('punkt_tab')


In [ ]:
# 라벨 변환
df['label'] = df['label'].map({'ham':0, 'spam': 1})

# text = ['abc','def'...] 얘는 100퍼 전부 다 들어감
# labels = [1,0,1]
texts, labels = df['message'].tolist(), df['label'].tolist()

# 문장 80,        문장 20,      레이블 80,      레이블 20
train_inputs, test_inputs, train_targets, test_targets = train_test_split(df['message'], df['label'], test_size=0.2, random_state=42)

# print(train_inputs)
# Series를 list로 변환하여 레이블 인덱스 문제를 제거합니다
# 얘는 80프로만 들어간 ['abc', 'bcd'...]형태
train_inputs_list = train_inputs.tolist()
train_targets_list = train_targets.tolist()
test_inputs_list = test_inputs.tolist()
test_targets_list = test_targets.tolist()

print(f"Number of documents: {len(texts)}")
print(f"Sample documents: \n{texts[0]}")

In [ ]:
from nltk.tokenize import word_tokenize
# "I love machine learning, it's amazing!"  띄어쓰기, 구두점, 기호로 나눔
# ['I', 'love', 'machine', 'learning', ',', 'it', "'s", 'amazing', '!']
# 메시지 길이 분포 확인
train_lengths = [len(word_tokenize(msg)) for msg in train_inputs]
test_lengths = [len(word_tokenize(msg)) for msg in test_inputs]

plt.hist(train_lengths, bins=30, alpha=0.7, label='Train Data')
plt.hist(test_lengths, bins=30, alpha=0.7, label='Test Data')
plt.xlabel('Message Length (words)')
plt.ylabel('Frequency')
plt.legend()
plt.title('Distribution of Message Lengths')
plt.show()


# 평균 및 최대 길이 확인
print(f"훈련 데이터 평균 길이: {np.mean(train_lengths)}")
print(f"훈련 데이터 최대 길이: {np.max(train_lengths)}")
print(f"훈련 데이터 중간값: {np.mean(train_lengths)}")



# Word2Vec

https://wikidocs.net/50739


In [ ]:
# 단어 토큰화
# 80퍼 문장 토큰화
train_sentences = [word_tokenize(msg) for msg in train_inputs]
# 20퍼 문장 토큰화
test_sentences = [word_tokenize(msg) for msg in test_inputs]

In [ ]:
!pip install gensim

In [23]:
from gensim.models import Word2Vec

# Word2Vec 모델 학습 및 단어-인덱스 매핑 생성
word2vec_model = Word2Vec(train_sentences, vector_size=50, window=5, min_count=1, sg=1)
# 모든 단어의 벡터를 하나의 행렬로 합치기
# +1은 인덱스 0을 “패딩용”으로 비워두는 거야
word2vec_matrix = np.zeros((len(word2vec_model.wv)+ 1, 50))
print(word2vec_matrix.shape) # 10180의 단어, 50개의 차원으로 나타낸 행렬임

# 이 모델의 각 단어를 꺼내서 word에 넣고 키: 번호로 딕셔너리 형태로 만들었음
# word2idx_word2vec = {"사랑": 1, "행복": 2}
word2idx_word2vec = {word: idx +1 for idx, word in enumerate(word2vec_model.wv.index_to_key)}

# 딕셔너리에 있는 아이템들을 word, idx로 뽑아서  딕셔너리에 있는 키값으로 인덱스를 뽑아서 행렬의 값으로 넣음
for word, idx in word2idx_word2vec.items():
    word2vec_matrix[idx] = word2vec_model.wv[word]

# word2vec_matrix[1] → '사랑' 벡터
# word2vec_matrix[2] → '행복' 벡터
# word2vec_matrix[3] → '고양이' 벡터
# 딥러닝 모델에서는 단어 대신에 숫자 인덱스를 받음


(10180, 50)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [ ]:
# 'he'이라는 단어의 벡터 확인
vector = word2vec_model.wv['he']
print(vector)
# 'he'과 가장 유사한 단어 5개 찾기
s_vector = word2vec_model.wv.most_similar('he', topn=5)
print(s_vector)
# 'he'과 'man'의 유사도 계산
s_vector_cal = word2vec_model.wv.similarity('he', 'man')
print(s_vector_cal)

In [ ]:
class WordSpamDataset(Dataset):
  """텍스트 문장을 토큰화, 정수 인코딩, 패딩/절단을 거쳐 지정된 길이(max_len)의 텐서로 변환하는 스팸 데이터셋"""
  def __init__(self, texts, labels, word2idx, max_len):
      self.texts = texts
      self.labels = labels
      self.word2idx = word2idx
      self.max_len = max_len

  def __len__(self):
      return len(self.texts)

  def __getitem__(self, idx):
      tokens = word_tokenize(self.texts[idx])
      encoded = [self.word2idx.get(word, 0) for word in tokens]  # OOV 단어는 0
      if len(encoded) < self.max_len:
          encoded += [0] * (self.max_len - len(encoded))
      else:
          encoded = encoded[:self.max_len]
      return torch.tensor(encoded, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float)

In [ ]:
class EmbeddingLSTM(nn.Module):
  """사전 학습된 임베딩과 LSTM을 사용하여 텍스트를 이진 분류하는 모델"""
  def __init__(self, embedding_matrix, hidden_dim, output_dim, num_layers=2, dropout=0.5):
      super(EmbeddingLSTM, self).__init__()
      num_embeddings, embedding_dim = embedding_matrix.shape
      self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix, dtype=torch.float).to(device), freeze=False)
      self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
      self.fc = nn.Linear(hidden_dim, output_dim)

  def forward(self, x):
      embedded = self.embedding(x)  # 임베딩 레이어를 통과
      _, (hidden, _) = self.lstm(embedded)  # LSTM의 출력
      output = self.fc(hidden[-1])  # 마지막 LSTM 층의 hidden state 사용
      return torch.sigmoid(output)  # sigmoid로 이진 분류

In [ ]:
embedding_dim = 50
max_len = 50
hidden_dim = 128
output_dim = 1

In [ ]:
# WordSpamDataset 및 데이터 로더
train_dataset_word2vec = WordSpamDataset(train_inputs_list, train_targets_list, word2idx_word2vec, max_len)
test_dataset_word2vec = WordSpamDataset(test_inputs_list, test_targets_list, word2idx_word2vec, max_len)

train_loader_word2vec = DataLoader(train_dataset_word2vec, batch_size=32, shuffle=True)
test_loader_word2vec = DataLoader(test_dataset_word2vec, batch_size=32, shuffle=False)

In [ ]:
def train(model, loader, criterion, optimizer):
  """모델을 1 epoch 동안 학습시키는 함수"""
  model.train()
  total_loss = 0
  for texts, labels in loader:
      texts, labels = texts.to(device), labels.to(device)
      labels = labels.unsqueeze(1)  # 라벨 크기를 (batch_size, 1)로 변환
      optimizer.zero_grad()
      outputs = model(texts)  # 모델 출력: (batch_size, 1)
      loss = criterion(outputs, labels)  # BCELoss 사용
      loss.backward()
      optimizer.step()
      total_loss += loss.item()
  return total_loss / len(loader)

def evaluate(model, loader):
  """모델의 성능(정확도)을 평가하는 함수"""
  model.eval()
  correct, total = 0, 0
  with torch.no_grad():
      for texts, labels in loader:
          texts, labels = texts.to(device), labels.to(device)
          labels = labels.unsqueeze(1).float()
          outputs = model(texts)
          # 확률값(outputs)이 0.5 이상이면 1(스팸), 아니면 0(햄)으로 변환
          predictions = (outputs >= 0.5).float()
          correct += (predictions == labels).sum().item()
          total += labels.size(0)
  return correct / total

In [ ]:
# 모델 초기화
model_word2vec = EmbeddingLSTM(word2vec_matrix, hidden_dim, output_dim).to(device)
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_word2vec.parameters(), lr=0.001)

In [ ]:
# 학습 및 평가 실행
for epoch in range(10):
    loss = train(model_word2vec, train_loader_word2vec, loss_fn, optimizer)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

accuracy = evaluate(model_word2vec, test_loader_word2vec)
print(f"Test Accuracy (Word2Vec with LSTM): {accuracy:.4f}")

# FastText


In [ ]:
from gensim.models import FastText

# FastText 모델 학습 및 단어-인덱스 매핑 생성
fasttext_model = FastText(sentences=train_sentences, vector_size=50, window=5, min_count=1, sg=1)
fasttext_matrix = np.zeros((len(fasttext_model.wv) + 1, 50))
word2idx_fasttext = {word: idx + 1 for idx, word in enumerate(fasttext_model.wv.index_to_key)}

for word, idx in word2idx_fasttext.items():
    fasttext_matrix[idx] = fasttext_model.wv[word]

In [ ]:
# FastText Dataset
train_dataset_fasttext = WordSpamDataset(train_inputs_list, train_targets_list, word2idx_fasttext, max_len)
test_dataset_fasttext = WordSpamDataset(test_inputs_list, test_targets_list, word2idx_fasttext, max_len)

# FastText DataLoader
train_loader_fasttext = DataLoader(train_dataset_fasttext, batch_size=32, shuffle=True)
test_loader_fasttext = DataLoader(test_dataset_fasttext, batch_size=32, shuffle=False)

In [ ]:
# 모델 초기화
model_fasttext = EmbeddingLSTM(fasttext_matrix, hidden_dim, output_dim).to(device)
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_fasttext.parameters(), lr=0.005)

In [ ]:
# 모델 학습
for epoch in range(10):
    loss = train(model_fasttext, train_loader_fasttext, loss_fn, optimizer)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

# 모델 평가
accuracy = evaluate(model_fasttext, test_loader_fasttext)
print(f"Test Accuracy: {accuracy:.4f}")

# GloVe
- `glove.6B.100d.txt` 사용


In [ ]:
import os
import urllib.request

# GloVe 다운로드 및 임베딩 로드
GLOVE_URL = "https://nlp.stanford.edu/data/glove.6B.zip"
GLOVE_ZIP = "glove.6B.zip"
GLOVE_FILE = "glove.6B.50d.txt"

if not os.path.exists(GLOVE_FILE):
    print("Downloading GloVe embeddings...")
    urllib.request.urlretrieve(GLOVE_URL, GLOVE_ZIP)
    import zipfile
    with zipfile.ZipFile(GLOVE_ZIP, 'r') as zip_ref:
        zip_ref.extractall()
    print("GloVe embeddings downloaded and extracted.")
else:
    print("GloVe embeddings already available.")

In [ ]:
embedding_dim = 50  # GloVe 임베딩 차원 설정
glove_embeddings = {}

with open(GLOVE_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coeffs = np.asarray(values[1:], dtype='float32')
        glove_embeddings[word] = coeffs

In [ ]:
len(glove_embeddings)

In [ ]:
# GloVe word2idx 생성
word2idx_glove = {word: idx + 1 for idx, word in enumerate(glove_embeddings.keys())}
glove_matrix = np.zeros((len(word2idx_glove) + 1, embedding_dim))

for word, idx in word2idx_glove.items():
    embedding_vector = glove_embeddings.get(word)
    if embedding_vector is not None:
        glove_matrix[idx] = embedding_vector

In [ ]:
# GloVe Dataset
train_dataset_glove = WordSpamDataset(train_inputs_list, train_targets_list, word2idx_glove, max_len)
test_dataset_glove = WordSpamDataset(test_inputs_list, test_targets_list, word2idx_glove, max_len)

# GloVe DataLoader
train_loader_glove = DataLoader(train_dataset_glove, batch_size=32, shuffle=True)
test_loader_glove = DataLoader(test_dataset_glove, batch_size=32, shuffle=False)

In [ ]:
model_glove = EmbeddingLSTM(glove_matrix, hidden_dim, output_dim).to(device)
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_glove.parameters(), lr=0.005)

# 모델 학습
for epoch in range(10):
    loss = train(model_glove, train_loader_glove, loss_fn, optimizer)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

# 모델 평가
accuracy = evaluate(model_glove, test_loader_glove)
print(f"Test Accuracy: {accuracy:.4f}")